# Browse Search — Main Table Exploration

## Purpose

Explore the raw eBay Browse Search dataset to understand its schema, data quality, record grain, and suitability for downstream product intelligence.

## Source

`gs://market-intelligence-raw/ebay/browse_search/`

## Scope

This notebook covers only the Browse Search main table.

Browse Search child tables are excluded from the current exploration scope.

## Exploration objectives

1. Understand the schema and field types.
2. Establish the current row count and distinct listing count.
3. Profile nulls and duplicate listing observations.
4. Examine categorical and temporal fields.
5. Document findings and open questions for downstream Bronze design.

In [0]:
# Load the dataset and inspect its current schema

from pyspark.sql import functions as F

browse_search_path = (
    "gs://market-intelligence-raw/ebay/browse_search/"
)

browse_search_df = (
    spark.read
    .option("multiLine", "false")
    .json(browse_search_path + "*.jsonl.gz")
)

print("Rows:", browse_search_df.count())
print("Columns:", len(browse_search_df.columns))

browse_search_df.printSchema()


# Dataset Loading and Initial Schema Inspection

## 1.1 Objective

The objective of this step is to load the raw Browse Search dataset from Google Cloud Storage (GCS) into a PySpark DataFrame and inspect its initial structure.

This establishes the baseline for subsequent data profiling, data-quality checks, and Bronze-to-Silver transformations.

## 1.2 Data Source and Loading Method

- **Source:** Google Cloud Storage (GCS)
- **Dataset:** eBay Browse Search
- **File format:** JSON Lines (compressed JSON files)
- **Storage location:** `gs://market-intelligence-raw/ebay/browse_search/`
- **Processing engine:** Apache Spark (PySpark)

The dataset is loaded using Spark's JSON reader with multiline parsing disabled. This is appropriate for JSON Lines data, where each line represents an individual JSON record.

The wildcard pattern `*.json.gz` allows Spark to read all matching compressed JSON files from the specified directory.

## 1.3 Initial Dataset Statistics

| Metric | Result |
|---|---:|
| Total Rows | 115,560 |
| Total Columns | 45 |
| File Format | JSON Lines (GZIP compressed) |

The initial snapshot contains 115,560 records and 45 columns.

These statistics establish the baseline against which subsequent transformations and data-quality checks can be compared.

## 1.4 Schema Inspection

The initial schema was inspected using `printSchema()`.

The schema contains the following data types:

| Data Type | Observed Columns |
|---|---|
| String | Listing identifiers, product attributes, dates, URLs, pricing fields, seller attributes, and marketplace information |
| Boolean | `adult_only`, `available_coupons`, `priority_listing`, `top_rated_buying_experience` |
| Long | `bid_count`, `seller__feedback_score` |

All 45 columns are nullable according to the inferred Spark schema.

### Important Observation: Pricing Fields Are Strings

Although several pricing fields represent numerical values, Spark has inferred them as strings.

Examples include:

- `price__value`
- `price__converted_from_value`
- `current_bid_price__value`
- `marketing_price__discount_amount__value`
- `marketing_price__original_price__value`
- `marketing_price__discount_percentage`

This means numerical operations such as price comparisons, aggregations, and opportunity scoring will require appropriate type conversion.

The conversion should be performed in the Silver layer after examining the actual values, handling nulls, and validating the expected currency and numeric formats.

### Important Observation: Date Fields Are Strings

The following listing lifecycle fields are also stored as strings:

- `item_creation_date`
- `item_end_date`
- `item_origin_date`

These fields will require timestamp or date parsing before performing date-based analysis, listing-age calculations, or time-based filtering.

The original raw values should remain available in the Bronze layer for traceability.

## 1.5 Initial Schema Assessment

The initial schema inspection establishes three important considerations for downstream processing:

1. **Data type standardization:** Convert numeric and date fields into appropriate Spark data types in Silver.
2. **Record identity:** Distinguish between ingestion-record identifiers (`_dlt_id`) and marketplace listing identifiers (`item_id`).
3. **Nested-field normalization:** The source contains flattened representations of nested objects, such as pricing, seller, and item-location attributes. These should be organized into logical groups during transformation and modeling.

## Conclusion

The raw Browse Search dataset has been successfully loaded into a PySpark DataFrame.

The initial snapshot contains 115,560 rows and 45 columns. The schema inspection identifies the primary business attributes, ingestion metadata, and fields requiring further type validation.

The next step is to investigate record grain, listing uniqueness, and repeated observations before defining the Silver-layer transformation strategy.


In [0]:
# Establish the record grain and listing uniqueness
browse_search_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("item_id").alias("distinct_item_ids"),
    F.sum(
        F.when(F.col("item_id").isNull(), 1).otherwise(0)
    ).alias("null_item_ids"),
    F.countDistinct("_dlt_id").alias("distinct_dlt_ids")
).show(truncate=False)

In [0]:
# how frequently listing IDs repeat:
item_id_counts = (
    browse_search_df
    .groupBy("item_id")
    .agg(F.count("*").alias("occurrences"))
)

display(
    item_id_counts
    .groupBy("occurrences")
    .agg(F.count("*").alias("number_of_item_ids"))
    .orderBy("occurrences")
)


# Record Grain and Listing Uniqueness Analysis

## 2.1 Objective

The objective of this step is to establish the record grain of the Browse Search dataset and determine whether each row represents a unique marketplace listing.

Two checks were performed:

1. Record-level uniqueness using `item_id` and `_dlt_id`.
2. Distribution of listing IDs based on their number of occurrences.

These checks help determine whether the dataset can be directly used for downstream analytical transformations or requires additional deduplication logic.

---

## 2.2 Record-Level Uniqueness

The following metrics were calculated using PySpark aggregation functions.

| Metric | Result |
|---|---:|
| Total Rows | 115,560 |
| Distinct Item IDs | 105,185 |
| Null Item IDs | 0 |
| Distinct `_dlt_id` | 115,560 |

### Findings

- **Listing identifier completeness:** There are no null values in `item_id`. Every record contains a marketplace listing identifier.

- **Listing identifier uniqueness:** There are 105,185 distinct item IDs across 115,560 records. Therefore, `item_id` is not a unique key in the current snapshot.

- **Ingestion record uniqueness:** All 115,560 records have distinct `_dlt_id` values. This confirms that every ingested record has a unique DLTHub record identifier.

- **Record grain:** The dataset contains multiple observations of certain marketplace listings. Consequently, the current record grain is an ingested observation rather than strictly one row per listing.

### Key Distinction

`item_id` and `_dlt_id` serve different purposes.

| Column | Purpose |
|---|---|
| `item_id` | Identifies a marketplace listing |
| `_dlt_id` | Uniquely identifies an ingested record |

A unique ingestion identifier does not guarantee a unique business entity.

---

## 2.3 Repeated Listing Observations

To understand the frequency of repeated listing IDs, the dataset was grouped by `item_id` and the number of occurrences per listing was calculated.

The resulting frequency distribution is shown below.

| Occurrences per Item ID | Number of Item IDs |
|---:|---:|
| 1 | 95,413 |
| 2 | 9,221 |
| 3 | 502 |
| 4 | 46 |
| 5 | 3 |
| **Total Distinct Item IDs** | **105,185** |

### Interpretation

- **Single-occurrence listings:** 95,413 item IDs appear exactly once. Most listings therefore have a single observation in the current snapshot.

- **Repeated listings:** 9,221 item IDs appear twice, while 502 appear three times. These repeated observations account for a subset of the dataset.

- **Higher-frequency repetitions:** 46 item IDs appear four times, and only 3 item IDs appear five times.

- **Overall observation pattern:** The distribution indicates that repeated listing observations exist, but their frequency is concentrated among listings with relatively few occurrences.

The frequency distribution is consistent with the possibility that the same listing was returned across multiple Browse Search results.

However, the occurrence counts alone do not establish why a listing was repeated.

---

## 2.4 Data Quality Assessment

Repeated `item_id` values should not automatically be classified as duplicate records or data-quality errors.

Browse Search is a discovery dataset. The same marketplace listing may appear in multiple search results, depending on the search queries and categories used during ingestion.

Therefore, multiple observations of the same listing may be valid source behavior.

The following distinctions are important:

| Scenario | Interpretation |
|---|---|
| Same `item_id`, different `_dlt_id` | Multiple ingested records associated with the same listing |
| Same listing returned by different search queries | Potentially valid repeated search observations |
| Same listing with changed pricing or attributes | Potentially represents a listing update |
| Identical listing payload ingested multiple times | Potential duplicate ingestion; requires further validation |

The current profiling results establish listing repetition, but do not independently determine which of these scenarios applies to every repeated record.

---

## 2.5 Implications for the Medallion Architecture

The findings influence how the dataset should be handled in subsequent Bronze, Silver, and Gold transformations.

### Bronze Layer

- Preserve the raw ingested observations.
- Retain `_dlt_id` and `_dlt_load_id` for ingestion traceability.
- Avoid prematurely removing repeated listing IDs, as they may represent valid source observations.

### Silver Layer

- Investigate repeated listing IDs and compare their business attributes.
- Standardize data types for pricing and listing dates, as identified during schema inspection.
- Define an explicit record grain and deduplication strategy based on the intended business use case.

### Gold Layer

- Build analytical tables according to the required business grain.
- For listing-level analysis, establish one representative record per listing using a documented selection rule.
- For historical or observation-level analysis, retain multiple observations where they provide meaningful analytical value.

Deduplication should be based on business requirements and validated record semantics, rather than simply dropping duplicate `item_id` values.

---

## 2.6 Final Conclusion

The initial Browse Search snapshot contains 115,560 ingested records, 105,185 distinct marketplace listing IDs, and no null listing identifiers.

Every record has a unique `_dlt_id`, but `item_id` is not unique. The dataset therefore contains multiple observations of certain listings.

Repeated listing observations are not automatically data-quality issues because Browse Search may return the same listing across different search results.

The record-grain investigation establishes the need to distinguish ingestion records from marketplace listings before implementing Silver-layer transformations.

**Outcome:** Initial schema inspection, record-level uniqueness checks, and repeated listing frequency analysis are complete. The findings provide the baseline for subsequent data engineering transformations and data-quality validation.
